In [79]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

import pickle

In [56]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

In [3]:
df=pd.read_csv("shoes_reviews.csv")
df.head()

,review_text,rating,label
0,Truly comfortable hiking boots. The attention ...,5,1
1,Can't go wrong with the classic Chuck!,5,0
2,Loved the shoe - super comfortable. Only probl...,3,0
3,LOOOOVE these. My 2 year old love these as we...,5,0
4,worst worst worst. Do not buy. worst.,1,1


In [5]:
print(df.shape)
df.isnull().sum()

(1821, 3)


review_text    0
rating         0
label          0
dtype: int64

In [6]:
df["label"].value_counts()

label
0    968
1    853
Name: count, dtype: int64

In [7]:
X=df["review_text"]
y=df["label"]

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
tokenizer=Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [16]:
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

print(X_train.iloc[0])
print(X_train_seq[0])

Solid pair of sneakers.
[540, 38, 14, 100]


In [17]:
review_lengths=[len(seq) for seq in X_train_seq]

print("Average length: ", np.mean(review_lengths))
print("Max length: ", np.max(review_lengths))

Average length:  31.9375
Max length:  393


In [48]:
max_length=120

X_train_pad=pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_pad=pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print(X_train_pad.shape)
print(X_test_pad.shape)

(1456, 120)
(365, 120)


In [41]:
vocab_size=len(tokenizer.word_index)+1
print("Vocabulary size: ", vocab_size)

Vocabulary size:  3529


In [61]:
model=Sequential([
    Input(shape=(max_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    Bidirectional(
        LSTM(64, dropout=0.3)
    ),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(1, activation='sigmoid')    
])

In [62]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [63]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ (None, 120, 128)            │         451,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_3 (Bidirectional)      │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 558,849 (2.13 MB)

 Trainable params: 558,849 (2.13 MB)

 Non-trainable params: 0 (0.00 B)

In [64]:
early_stop=EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [65]:
history=model.fit(
    X_train_pad,
    y_train,

    epochs=10,
    batch_size=32,

    validation_data=(X_test_pad, y_test),

    callbacks=[early_stop]
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.7589 - loss: 0.5053 - val_accuracy: 0.8932 - val_loss: 0.2728
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9451 - loss: 0.1501 - val_accuracy: 0.9616 - val_loss: 0.0986
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9718 - loss: 0.0690 - val_accuracy: 0.9671 - val_loss: 0.1039
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.9870 - loss: 0.0425 - val_accuracy: 0.9589 - val_loss: 0.1257
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9883 - loss: 0.0326 - val_accuracy: 0.9562 - val_loss: 0.1474
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.9952 - loss: 0.0199 - val_accuracy: 0.9507 - val_loss: 0.1719


In [66]:
loss, accuracy=model.evaluate(X_test_pad, y_test)

print("Test loss: ", loss)
print("Test accuracy: ", accuracy)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9616 - loss: 0.0986
Test loss:  0.0985996425151825
Test accuracy:  0.9616438150405884


In [72]:
y_pred_probs=model.predict(X_test_pad)
y_pred=(y_pred_probs>0.5).astype(int)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


In [73]:
cm=confusion_matrix(y_test, y_pred)
print(cm)

[[179  12]
 [  2 172]]


In [74]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.94      0.96       191
           1       0.93      0.99      0.96       174

    accuracy                           0.96       365
   macro avg       0.96      0.96      0.96       365
weighted avg       0.96      0.96      0.96       365



In [75]:
def predict_review(review):

    # lowercase
    # review = review.lower()

    # convert to sequence
    sequence = tokenizer.texts_to_sequences([review])

    # pad
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    # predict
    prediction = model.predict(padded)[0][0]

    # display
    print("Prediction Score:", prediction)

    if prediction > 0.5:
        print("Fake Review")
    else:
        print("Real Review")

In [77]:
predict_review(
    "These shoes are very comfortable and lightweight. I've been using them daily for walks and they still feel great after a few weeks."
)
predict_review(
    "Absolutely the greatest shoes ever made in human history! Perfect comfort, perfect quality, perfect everything! Buy immediately!!!"
)
predict_review(
    "Amazing shoes. Best ever."
)
predict_review(
    "Good fit overall. The design looks nice and they're comfortable, although the sole could have slightly better grip."
)
predict_review(
    "Worst product ever. Horrible quality. Complete waste of money. Never buy this trash."
)
predict_review(
    "The shoes looked good initially, but after two weeks the stitching started coming loose near the sides."
)
predict_review(
    "Premium imported luxury sports sneaker with ultimate comfort and stylish fashion design for all occasions."
)
predict_review(
    "Delivery was on time and the shoes fit as expected. Nothing exceptional but decent for the price."
)
predict_review(
    "Very very very amazing shoes. Super super comfortable and fantastic quality. Totally recommend."
)
predict_review(
    "The shoes are comfortable for casual wear, but I don't think they are suitable for long-distance running."
)
predict_review(
    "Excellent product with excellent quality and excellent comfort. Highly highly recommended."
)
predict_review(
    "Terrible. Worst shoes ever."
)
predict_review(
    "I bought these for my morning walks and they've been holding up well so far. The cushioning feels decent and sizing was accurate."
)
predict_review(
    "Must buy immediately! Best premium shoes online right now! 100% satisfaction guaranteed!"
)
predict_review(
    "Really comfortable shoes with stylish design. Very happy with the purchase and quality."
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction Score: 0.0034964685
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction Score: 0.981575
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction Score: 0.98498726
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction Score: 0.0055467384
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Prediction Score: 0.9949798
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction Score: 0.111406125
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Prediction Score: 0.9821147
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction Score: 0.06250239
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Prediction Score: 0.25363886
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction Score: 0.036798585
Real Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction Score: 0.98301363
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction Score: 0.990104
Fake Review
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Prediction Sco

In [78]:
model.save("fake_review_bilstm_v1.keras")

In [82]:
with open("tokenizer.pkl", "wb") as f:  #save tokenizer
    pickle.dump(tokenizer, f)